In [2]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np

In [3]:
# Load RNA and ATAC
rna = sc.read_h5ad("040_singlets_gex.h5ad")
atac = sc.read_h5ad("040_singlets_atac.h5ad")

# Basic information
print("RNA shape (cells x genes):", rna.shape)
print("ATAC shape (cells x peaks):", atac.shape)

RNA shape (cells x genes): (4724, 36601)
ATAC shape (cells x peaks): (4724, 6062095)


In [4]:
# Align cells (barcodes should match)
common = rna.obs_names.intersection(atac.obs_names)
rna = rna[common, :]
atac = atac[common, :]
atac = atac[rna.obs_names, :]   # same order

In [5]:
# Build combined AnnData
adata = ad.AnnData(X=rna.X, obs=rna.obs, var=rna.var)
adata.obsm['ATAC'] = atac.X           # raw peak matrix (cells × peaks)
# For training script compatibility, also store as 'ATAC_gene' (same raw peaks)
adata.obsm['ATAC_gene'] = atac.X

In [6]:
# Transfer guide annotation
adata.obs['guide_target'] = rna.obs['guide_target']

In [7]:
# Compute pseudotime from RNA (first diffusion component)
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000, flavor='seurat')
sc.tl.pca(adata, n_comps=30, use_highly_variable=True)
sc.pp.neighbors(adata)
sc.tl.diffmap(adata, n_comps=15)
pseudotime = adata.obsm['X_diffmap'][:, 0]
pseudotime = (pseudotime - pseudotime.min()) / (pseudotime.max() - pseudotime.min())
adata.obs['pseudotime'] = pseudotime

C:\Users\phabe\miniconda3\envs\tmo_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Save merged object
adata.write("multip_seq_combined.h5ad", compression='gzip')

In [9]:
import anndata as ad
adata = ad.read_h5ad("multip_seq_combined.h5ad")
print("Shape:", adata.shape)          # Expect (4724, n_genes)
print("Has .obsm['ATAC']?", 'ATAC' in adata.obsm)
print("Has .obsm['ATAC_gene']?", 'ATAC_gene' in adata.obsm)
print("Has .obs['guide_target']?", 'guide_target' in adata.obs)
print("Has .obs['pseudotime']?", 'pseudotime' in adata.obs)
print("Control cells (NTC):", sum(adata.obs['guide_target'] == 'NTC'))
print("SMARCB1 cells:", sum(adata.obs['guide_target'] == 'SMARCB1'))

Shape: (4724, 36601)
Has .obsm['ATAC']? True
Has .obsm['ATAC_gene']? True
Has .obs['guide_target']? True
Has .obs['pseudotime']? True
Control cells (NTC): 1144
SMARCB1 cells: 147


In [10]:
adata_ntc = adata[adata.obs['guide_target'] == 'NTC'].copy()
adata_ntc.write("ntc_only.h5ad")
print(f"Control cells: {adata_ntc.n_obs}")

Control cells: 1144


In [2]:
import anndata as ad
import pandas as pd

# Load your combined AnnData (adjust the path)
adata = ad.read_h5ad("multip_seq_combined.h5ad")

# Print first 20 gene names
print("First 20 gene names:")
print(adata.var_names[:20])

# Search for known SMARCB1 target-related genes
candidates = ['SMARCB1', 'CDKN1A', 'MYC', 'E2F1', 'CCND1', 'CDK2', 'CDK4', 'ARID1A', 'SMARCA4']
print("\nChecking for candidate genes:")
for gene in candidates:
    if gene in adata.var_names:
        print(f"  Found: {gene}")
    else:
        # Try case‑insensitive partial match
        matches = [g for g in adata.var_names if gene.lower() in g.lower()]
        if matches:
            print(f"  Possibly {gene}? (matches: {matches[:3]}...)")
        else:
            print(f"  Not found: {gene}")

# Optional: save all gene names to a text file
with open("gene_names.txt", "w") as f:
    for gene in adata.var_names:
        f.write(gene + "\n")
print("Full gene list saved to gene_names.txt")

First 20 gene names:
Index(['MIR1302-2HG', 'FAM138A', 'OR4F5', 'AL627309.1', 'AL627309.3',
       'AL627309.2', 'AL627309.5', 'AL627309.4', 'AP006222.2', 'AL732372.1',
       'OR4F29', 'AC114498.1', 'OR4F16', 'AL669831.2', 'LINC01409', 'FAM87B',
       'LINC01128', 'LINC00115', 'FAM41C', 'AL645608.6'],
      dtype='object')

Checking for candidate genes:
  Found: SMARCB1
  Found: CDKN1A
  Found: MYC
  Found: E2F1
  Found: CCND1
  Found: CDK2
  Found: CDK4
  Found: ARID1A
  Found: SMARCA4
Full gene list saved to gene_names.txt
